In [ ]:
import os
import zipfile
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry
import pandas as pd

# ضع هنا رابط التحميل المباشر للملف المضغوط (.zip)
ZIP_URL = "https://data.mendeley.com/public-api/zip/2fgpn5dwgc/download/3"
WORK_DIR = "/content/zip_dataset"
EXTRACT_DIR = os.path.join(WORK_DIR, "extracted")

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(EXTRACT_DIR, exist_ok=True)

zip_path = os.path.join(WORK_DIR, "dataset.zip")

# إعداد استراتيجية إعادة المحاولة عند حدوث أخطاء السيرفر المؤقتة مثل 502
session = requests.Session()
retries = Retry(
    total=5,
    backoff_factor=2,
    status_forcelist=[500, 502, 503, 504]
)
session.mount("https://", HTTPAdapter(max_retries=retries))

print("جاري تحميل الملف المضغوط...")
try:
    with session.get(ZIP_URL, stream=True, timeout=120) as resp:
        resp.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    print(f"اكتمل التحميل. الحجم: {os.path.getsize(zip_path) / (1024 * 1024):.2f} MB")
except requests.exceptions.HTTPError as e:
    print(f"حدث خطأ أثناء التحميل: {e}")
    print("تنبيه: يبدو أن سيرفر الموقع لا يزال يواجه مشاكل (502). يرجى محاولة تشغيل الخلية لاحقاً أو التحقق من الرابط.")

# التحقق من أن الملف ZIP سليم وفك ضغط ملفات CSV فقط
csv_files = []
if os.path.exists(zip_path) and zipfile.is_zipfile(zip_path):
    print("جاري فك ضغط ملفات CSV من الأرشيف...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.namelist():
            if member.lower().endswith(".csv") and not member.startswith("__MACOSX"):
                zf.extract(member, EXTRACT_DIR)
                csv_files.append(os.path.join(EXTRACT_DIR, member))
    print("تم استخراج ملفات CSV:", csv_files)
else:
    if os.path.exists(zip_path):
        print("تنبيه: الملف المحمّل ليس بصيغة ZIP صالحة.")

if csv_files:
    df = pd.read_csv(csv_files[0])
    print("أبعاد أول ملف CSV:", df.shape)
    print("الأعمدة:", df.columns.tolist())

In [ ]:
import os
import shutil
from google.colab import drive

# 1. ربط قوقل درايف ببيئة العمل
print("جاري ربط Google Drive...")
drive.mount('/content/drive')

# 2. تحديد المسار المصدر والمسار الهدف في قوقل درايف
# سنقوم بنسخ المجلد المستخرج بالكامل
SOURCE_DIR = "/content/zip_dataset/extracted"
DESTINATION_DIR = "/content/drive/MyDrive/ASLAD_Dataset"

# 3. التحقق من وجود المجلد المصدر وبدء عملية النسخ
if os.path.exists(SOURCE_DIR):
    print(f"جاري نسخ المجلد من {SOURCE_DIR} إلى {DESTINATION_DIR}...")

    # إذا كان المجلد موجوداً مسبقاً في الدرايف، سنقوم بحذفه أولاً لتجنب التكرار والتداخل
    if os.path.exists(DESTINATION_DIR):
        print("المجلد موجود مسبقاً في الدرايف، يتم تحديثه الآن...")
        shutil.rmtree(DESTINATION_DIR)

    shutil.copytree(SOURCE_DIR, DESTINATION_DIR)
    print("تم تحميل ونسخ المجلد بنجاح إلى Google Drive في المسار: ASLAD_Dataset")
else:
    print("تنبيه: لم يتم العثور على المجلد المستخرج. يرجى التأكد من تشغيل الخلية السابقة بنجاح أولاً.")

In [ ]:
import os
import pandas as pd

DATASET_DIR = "/content/drive/MyDrive/ASLAD_Dataset"

csv_files = []
subfolders = []

for root, dirs, files in os.walk(DATASET_DIR):
    for d in sorted(dirs):
        subfolders.append(os.path.join(root, d))
    for f in sorted(files):
        full_path = os.path.join(root, f)
        if f.lower().endswith(".csv"):
            size_mb = os.path.getsize(full_path) / (1024 * 1024)
            csv_files.append((full_path, size_mb))

print(f"عدد المجلدات الفرعية: {len(subfolders)}")
print("أول 10 مجلدات فرعية:", subfolders[:10])
print("\nملفات CSV المكتشفة:")
for path, size_mb in csv_files:
    print(f"  - {path} ({size_mb:.2f} MB)")

In [ ]:
import pandas as pd

CSV_PATH = (
    "/content/drive/MyDrive/ASLAD_Dataset/"
    "ASLAD-190K Arabic Sign Language Alphabet Dataset c/"
    "ASLAD-96K/ASLAD-96K_63Hand landmarks+26Angles+Label+ImgName.csv"
)

df = pd.read_csv(CSV_PATH)

print("عدد الصفوف وعدد الأعمدة:", df.shape)
print("أسماء أول 5 أعمدة:", df.columns[:5].tolist())
print("أسماء آخر 6 أعمدة:", df.columns[-6:].tolist())

# عرض آخر 5 أعمدة لأول 10 صفوف لمشاهدة Label و ImgName
last_cols = df.columns[-5:].tolist()
df[last_cols].head(10)

In [ ]:
# البحث عن الأعمدة بالأسماء أولاً
target_col = next((c for c in df.columns if c.lower() in ["label", "class"]), None)
img_col = next((c for c in df.columns if "img" in c.lower() or "name" in c.lower() or "file" in c.lower()), None)

# إذا لم يتم العثور عليها بالاسم، نستخدم العمودين الأخيرين بناءً على هيكل البيانات الفعلي
if target_col is None and len(df.columns) >= 2:
    target_col = df.columns[-2]  # العمود قبل الأخير (مثلاً 4190)
if img_col is None and len(df.columns) >= 1:
    img_col = df.columns[-1]   # العمود الأخير (مثلاً 4191)

print("عمود التصنيف المكتشف:", target_col)
print("عمود اسم الصورة المكتشف:", img_col)

if target_col is not None:
    print("عدد الفئات الفريدة:", df[target_col].nunique())
    print("الفئات المرتبة:", sorted(df[target_col].dropna().unique().tolist()))

# عرض عينة من أسماء الصور لكل فئة
if img_col and target_col:
    print("\nأمثلة على الصور لكل فئة:")
    for lbl, grp in df.groupby(target_col):
        sample_imgs = grp[img_col].dropna().head(3).tolist()
        print(f"Label = {lbl} | عدد الصفوف = {len(grp)} | أمثلة: {sample_imgs}")

In [ ]:
import re
import pandas as pd

# نفترض أن df محمّل مسبقاً من الخلية 1
print("إجمالي الأعمدة:", len(df.columns))
print("أول 10 أعمدة:", df.columns[:10].tolist())
print("الأعمدة من 85 إلى النهاية:", df.columns[85:].tolist())

label_col = df.columns[4190] if isinstance(4190, int) and "4190" not in df.columns else "4190"
img_col = df.columns[4191] if isinstance(4191, int) and "4191" not in df.columns else "4191"

# إذا كانت أسماء الأعمدة في df نصوصاً مثل '4190' و '4191' نستخدمها مباشرة:
if "4190" in df.columns and "4191" in df.columns:
    label_col, img_col = "4190", "4191"

def extract_sign_base(name: str) -> str:
    if not isinstance(name, str):
        return ""
    base = re.sub(r"\s*\(\d+\)\.[a-zA-Z]+$", "", name.strip())
    return base.strip()

mapping_summary = []
for lbl, grp in df.groupby(label_col):
    bases = grp[img_col].dropna().apply(extract_sign_base).value_counts().to_dict()
    mapping_summary.append({"Label": lbl, "Count": len(grp), "Sign_Names": bases})

df_map = pd.DataFrame(mapping_summary)
display(df_map)

In [ ]:
# قاموس مبدئي لربط رقم الفئة بالاسم الإنجليزي والحرف العربي المقابل
# راجعه مع ASLAD-190K_Letters.jpg للتأكد التام من الرموز الخاصة مثل toot و al و la
label_info = {
    0:  ("ain", "ع"),
    1:  ("al", "الـ"),
    2:  ("gaaf", "ق"),
    3:  ("bb", "ب"),
    4:  ("dal", "د"),
    5:  ("dhad", "ض"),
    6:  ("aleff", "أ"),
    7:  ("dha", "ظ"),
    8:  ("ha", "هـ"),
    9:  ("fa", "ف"),
    10: ("ghain", "غ"),
    11: ("haa+", "ح"),
    12: ("jeem+", "ج"),
    13: ("kaaf", "ك"),
    14: ("ra", "ر"),
    15: ("meem", "م"),
    16: ("nun+", "ن"),
    17: ("sheen", "ش"),
    18: ("laam++", "ل"),
    19: ("khaa+", "خ"),
    20: ("seen", "س"),
    21: ("saad", "ص"),
    22: ("la", "لا"),
    23: ("ta+", "ط"),
    24: ("waw", "و"),
    25: ("zay", "ز"),
    26: ("yaa", "ي"),
    27: ("taa", "ت"),
    28: ("thaa", "ث"),
    29: ("ya+", "ى / ي"),
    30: ("thal+", "ذ"),
    31: ("toot", "ة"),
}

# اختبار سريع لترجمة أول 10 صفوف في الجدول
for i in range(10):
    lbl_val = int(df.loc[i, label_col])
    img_val = df.loc[i, img_col]
    eng_name, ar_char = label_info[lbl_val]
    print(f"صف {i} | Label={lbl_val} | {eng_name} ({ar_char}) | الصورة: {img_val}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# تحديد عمودي الفئة واسم الصورة واستخراج أعمدة الميزات الرقمية
target_col = "4190" if "4190" in df.columns else df.columns[-2]
img_col = "4191" if "4191" in df.columns else df.columns[-1]
feature_cols = [c for c in df.columns if c not in [target_col, img_col]]

df_work = df.copy()
print("عدد الصفوف قبل التنظيف:", len(df_work))

# 1. تحويل أعمدة الميزات إلى أرقام واستبعاد القيم غير الرقمية أو غير المحدودة (NaN, inf, -inf)
df_work[feature_cols] = (
    df_work[feature_cols]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)
valid_mask = df_work[feature_cols].notna().all(axis=1) & df_work[target_col].notna()
df_work = df_work.loc[valid_mask].copy()
print("بعد استبعاد القيم المفقودة/غير المحدودة:", len(df_work))

# 2. استبعاد الصفوف الصفرية بالكامل أو ذات القيم الشاذة المتطرفة خارج النطاق المنطقي
# (حد واسع للإحداثيات والزوايا لتجنب حذف العينات السليمة مع استبعاد القيم التالفة بوضوح)
non_zero_mask = (df_work[feature_cols].abs().sum(axis=1) > 1e-8)
extreme_mask = (df_work[feature_cols].abs() > 500.0).any(axis=1)
df_work = df_work.loc[non_zero_mask & ~extreme_mask].copy()
print("بعد استبعاد الصفوف الصفرية والقيم الشاذة المتطرفة:", len(df_work))

# 3. إزالة التكرار على مستوى متجه الميزات الكامل قبل التقسيم
df_work = df_work.drop_duplicates(subset=feature_cols, keep="first").reset_index(drop=True)
print("بعد إزالة التكرار في الميزات:", len(df_work))

# 4. تقسيم البيانات (70% تدريب، 15% تحقق، 15% اختبار) مع الحفاظ على توزيع الفئات
train_df, temp_df = train_test_split(
    df_work,
    test_size=0.30,
    random_state=42,
    stratify=df_work[target_col],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df[target_col],
)

# 5. التحقق الصريح من عدم وجود أي تداخل أو تكرار بين المجموعات الثلاث
train_feat_keys = set(pd.util.hash_pandas_object(train_df[feature_cols], index=False))
val_feat_keys = set(pd.util.hash_pandas_object(val_df[feature_cols], index=False))
test_feat_keys = set(pd.util.hash_pandas_object(test_df[feature_cols], index=False))

overlap_train_test_feat = len(train_feat_keys & test_feat_keys)
overlap_train_val_feat = len(train_feat_keys & val_feat_keys)
overlap_val_test_feat = len(val_feat_keys & test_feat_keys)

overlap_train_test_img = len(set(train_df[img_col]) & set(test_df[img_col]))

print(f"\nأحجام المجموعات -> التدريب: {len(train_df)} | التحقق: {len(val_df)} | الاختبار: {len(test_df)}")
print(
    f"تداخل الميزات -> تدريب/اختبار: {overlap_train_test_feat} | "
    f"تدريب/تحقق: {overlap_train_val_feat} | تحقق/اختبار: {overlap_val_test_feat}"
)
print(f"أسماء صور مشتركة بين التدريب والاختبار: {overlap_train_test_img}")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1. إعداد البذور العشوائية والجهاز
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("الجهاز المستخدم للتدريب:", device)

# 2. تجهيز الميزات X والترميز الرقمي للفئات y
X_train = train_df[feature_cols].values.astype(np.float32)
X_val = val_df[feature_cols].values.astype(np.float32)
X_test = test_df[feature_cols].values.astype(np.float32)

le = LabelEncoder()
y_train = le.fit_transform(train_df[target_col].values)
y_val = le.transform(val_df[target_col].values)
y_test = le.transform(test_df[target_col].values)
num_classes = len(le.classes_)

# ملاءمة StandardScaler على بيانات التدريب فقط لمنع تسرب المعلومات
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train_scaled), torch.tensor(y_train, dtype=torch.long)),
    batch_size=256,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(torch.tensor(X_val_scaled), torch.tensor(y_val, dtype=torch.long)),
    batch_size=512,
    shuffle=False,
)
test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test_scaled), torch.tensor(y_test, dtype=torch.long)),
    batch_size=512,
    shuffle=False,
)


# 3. تعريف معمارية MLP
class SignMLP(nn.Module):
    def __init__(self, in_features: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


model = SignMLP(in_features=X_train.shape[1], num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

# 4. حلقة التدريب والاختيار حسب أفضل دقة تحقق (Validation Accuracy)
EPOCHS = 30
best_val_acc = 0.0
best_state_dict = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(by)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for bx, by in val_loader:
            logits = model(bx.to(device))
            val_preds.extend(logits.argmax(dim=1).cpu().numpy())
            val_targets.extend(by.numpy())

    val_acc = accuracy_score(val_targets, val_preds)
    val_f1 = f1_score(val_targets, val_preds, average="macro")
    scheduler.step(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(
            f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | "
            f"Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f} | Best Val Acc: {best_val_acc:.4f}"
        )

# 5. التقييم النهائي على مجموعة الاختبار (Test Set) المعزولة
model.load_state_dict({k: v.to(device) for k, v in best_state_dict.items()})
model.eval()
test_preds, test_targets = [], []
with torch.no_grad():
    for bx, by in test_loader:
        logits = model(bx.to(device))
        test_preds.extend(logits.argmax(dim=1).cpu().numpy())
        test_targets.extend(by.numpy())

test_acc = accuracy_score(test_targets, test_preds)
test_f1 = f1_score(test_targets, test_preds, average="macro")

print(f"\nالنتيجة النهائية على مجموعة الاختبار -> Accuracy: {test_acc:.4f} | Macro-F1: {test_f1:.4f}")
print("\nتقرير التصنيف التفصيلي:")
print(classification_report(test_targets, test_preds, target_names=[str(c) for c in le.classes_], digits=3))

In [ ]:
import os
import json
import pickle
import torch

SAVE_DIR = "/content/drive/MyDrive/ASLAD_Dataset/arsl_mlp_bundle"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. حفظ أوزان أفضل نموذج ومعلومات البنية
weights_to_save = best_state_dict if "best_state_dict" in globals() and best_state_dict is not None else model.state_dict()
checkpoint = {
    "model_state_dict": {k: v.cpu() for k, v in weights_to_save.items()},
    "in_features": int(X_train.shape[1]),
    "num_classes": int(num_classes),
    "feature_cols": list(feature_cols),
}
model_path = os.path.join(SAVE_DIR, "sign_mlp_checkpoint.pt")
torch.save(checkpoint, model_path)

# 2. حفظ StandardScaler و LabelEncoder
scaler_path = os.path.join(SAVE_DIR, "standard_scaler.pkl")
encoder_path = os.path.join(SAVE_DIR, "label_encoder.pkl")

with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)

with open(encoder_path, "wb") as f:
    pickle.dump(le, f)

# 3. حفظ قاموس الربط بالحروف العربية كمرجع مساعد
if "label_to_arabic" in globals():
    map_path = os.path.join(SAVE_DIR, "label_to_arabic.json")
    with open(map_path, "w", encoding="utf-8") as f:
        json.dump(label_to_arabic, f, ensure_ascii=False, indent=2)

# 4. تحقق سريع من صحة الملفات المحفوظة بإعادة قراءتها
loaded_ckpt = torch.load(model_path, map_location="cpu")
with open(scaler_path, "rb") as f:
    loaded_scaler = pickle.load(f)
with open(encoder_path, "rb") as f:
    loaded_le = pickle.load(f)

print("تم الحفظ بنجاح في المجلد:", SAVE_DIR)
print("عدد الميزات المحفوظة:", loaded_ckpt["in_features"], "| عدد الفئات:", loaded_ckpt["num_classes"])
print("حجم ملف النموذج (KB):", round(os.path.getsize(model_path) / 1024, 2))
print("حجم ملف Scaler (KB):", round(os.path.getsize(scaler_path) / 1024, 2))
print("حجم ملف Encoder (KB):", round(os.path.getsize(encoder_path) / 1024, 2))

In [ ]:
import os
from google.colab import files

# تحديد المجلد الذي يحتوي على الملفات المحفوظة
SAVE_DIR = "/content/drive/MyDrive/ASLAD_Dataset/arsl_mlp_bundle"

# مسارات الملفات الثلاثة
files_to_download = [
    os.path.join(SAVE_DIR, "sign_mlp_checkpoint.pt"),
    os.path.join(SAVE_DIR, "standard_scaler.pkl"),
    os.path.join(SAVE_DIR, "label_encoder.pkl")
]

# تنزيل كل ملف
print("جاري بدء تنزيل الملفات...")
for file_path in files_to_download:
    if os.path.exists(file_path):
        print(f"تنزيل: {os.path.basename(file_path)}")
        files.download(file_path)
    else:
        print(f"تنبيه: الملف غير موجود في المسار المحدد: {file_path}")